In [1]:
import pandas as pd
import os
from pathlib import Path

def compile_mobility_metrics(river_list_csv, output_csv):
    """
    Compile all individual river mobility metrics into a single master CSV.
    
    Parameters:
        river_list_csv (str): Path to CSV containing river_name and working_directory columns
                             (e.g., Greenbergetal2024_river_datasheet.csv)
        output_csv (str): Path where the compiled master CSV will be saved
    
    Returns:
        pd.DataFrame: Combined mobility metrics for all rivers
    """
    # Read the list of rivers
    river_data = pd.read_csv(river_list_csv)
    
    # List to store all mobility dataframes
    all_metrics = []
    
    # Track statistics
    found_count = 0
    missing_count = 0
    missing_rivers = []
    
    print("Compiling mobility metrics from individual river CSVs...")
    print("=" * 80)
    
    # Iterate through each river
    for _, row in river_data.iterrows():
        river_name = row['river_name']
        working_directory = row['working_directory']
        
        # Construct path to this river's mobility metrics CSV
        mobility_csv_path = os.path.join(
            working_directory,
            "RiverMapping",
            "Mobility",
            river_name,
            f"{river_name}_mobility_metrics.csv"
        )
        
        # Check if file exists
        if os.path.exists(mobility_csv_path):
            try:
                # Read the mobility metrics
                df = pd.read_csv(mobility_csv_path)
                
                # Add river_name column as the first column
                df.insert(0, 'river_name', river_name)
                
                # Append to list
                all_metrics.append(df)
                found_count += 1
                
                print(f"✓ {river_name}: {len(df)} reaches")
                
            except Exception as e:
                print(f"✗ {river_name}: Error reading CSV - {e}")
                missing_rivers.append(river_name)
                missing_count += 1
        else:
            print(f"✗ {river_name}: Mobility CSV not found")
            missing_rivers.append(river_name)
            missing_count += 1
    
    print("\n" + "=" * 80)
    print(f"SUMMARY:")
    print(f"  Rivers with mobility metrics: {found_count}")
    print(f"  Rivers missing metrics: {missing_count}")
    
    if missing_rivers:
        print(f"\nMissing rivers:")
        for river in missing_rivers:
            print(f"  - {river}")
    
    # Combine all dataframes
    if all_metrics:
        master_df = pd.concat(all_metrics, ignore_index=True)
        
        # Ensure columns are in the desired order
        desired_columns = [
            'river_name',
            'ds_order',
            'Tr_yr',
            'Tm_yr',
            'Tm_over_Tr',
            'Tw_yr',
            'Cr_peryr',
            'Cm_peryr',
            'median_Aw_m2',
            'Pr_over_Aw',
            'Pm_over_Aw',
            'Pswitch'
        ]
        
        # Include any additional columns that might be present (e.g., fit methods)
        additional_cols = [col for col in master_df.columns if col not in desired_columns]
        final_columns = desired_columns + additional_cols
        
        # Reorder columns (only include columns that exist)
        existing_columns = [col for col in final_columns if col in master_df.columns]
        master_df = master_df[existing_columns]
        
        # Sort by river_name and ds_order
        master_df = master_df.sort_values(['river_name', 'ds_order'])
        
        # Save to CSV
        master_df.to_csv(output_csv, index=False)
        
        print(f"\n✓ Master CSV saved to: {output_csv}")
        print(f"  Total reaches: {len(master_df)}")
        print(f"  Total rivers: {master_df['river_name'].nunique()}")
        print(f"  Columns: {list(master_df.columns)}")
        
        # Display summary statistics
        print("\n" + "=" * 80)
        print("SUMMARY STATISTICS:")
        print(f"  Tr range: {master_df['Tr_yr'].min():.2f} - {master_df['Tr_yr'].max():.2f} years")
        print(f"  Tm range: {master_df['Tm_yr'].min():.2f} - {master_df['Tm_yr'].max():.2f} years")
        print(f"  Tm/Tr range: {master_df['Tm_over_Tr'].min():.3f} - {master_df['Tm_over_Tr'].max():.3f}")
        print(f"  Pswitch range: {master_df['Pswitch'].min():.3f} - {master_df['Pswitch'].max():.3f}")
        
        return master_df
    else:
        print("\n✗ No mobility metrics found. Cannot create master CSV.")
        return None


def compile_subset(river_list_csv, output_csv, river_subset=None):
    """
    Compile mobility metrics for a subset of rivers.
    
    Parameters:
        river_list_csv (str): Path to CSV containing river_name and working_directory
        output_csv (str): Output path for compiled CSV
        river_subset (list): List of river names to include. If None, includes all.
    
    Returns:
        pd.DataFrame: Combined mobility metrics
    """
    # Read full river list
    river_data = pd.read_csv(river_list_csv)
    
    # Filter to subset if provided
    if river_subset is not None:
        river_data = river_data[river_data['river_name'].isin(river_subset)]
        print(f"Processing subset of {len(river_data)} rivers")
    
    # Use main compilation function
    return compile_mobility_metrics_from_df(river_data, output_csv)


def compile_mobility_metrics_from_df(river_df, output_csv):
    """
    Helper function that works with a dataframe instead of reading from CSV.
    """
    all_metrics = []
    found_count = 0
    missing_count = 0
    missing_rivers = []
    
    print("Compiling mobility metrics...")
    print("=" * 80)
    
    for _, row in river_df.iterrows():
        river_name = row['river_name']
        working_directory = row['working_directory']
        
        mobility_csv_path = os.path.join(
            working_directory,
            "RiverMapping",
            "Mobility",
            river_name,
            f"{river_name}_mobility_metrics.csv"
        )
        
        if os.path.exists(mobility_csv_path):
            try:
                df = pd.read_csv(mobility_csv_path)
                df.insert(0, 'river_name', river_name)
                all_metrics.append(df)
                found_count += 1
                print(f"✓ {river_name}: {len(df)} reaches")
            except Exception as e:
                print(f"✗ {river_name}: Error - {e}")
                missing_rivers.append(river_name)
                missing_count += 1
        else:
            print(f"✗ {river_name}: Not found")
            missing_rivers.append(river_name)
            missing_count += 1
    
    print(f"\nFound: {found_count}, Missing: {missing_count}")
    
    if all_metrics:
        master_df = pd.concat(all_metrics, ignore_index=True)
        
        desired_columns = [
            'river_name', 'ds_order', 'Tr_yr', 'Tm_yr', 'Tm_over_Tr', 
            'Tw_yr', 'Cr_peryr', 'Cm_peryr', 'median_Aw_m2', 
            'Pr_over_Aw', 'Pm_over_Aw', 'Pswitch'
        ]
        
        additional_cols = [col for col in master_df.columns if col not in desired_columns]
        final_columns = desired_columns + additional_cols
        existing_columns = [col for col in final_columns if col in master_df.columns]
        
        master_df = master_df[existing_columns]
        master_df = master_df.sort_values(['river_name', 'ds_order'])
        master_df.to_csv(output_csv, index=False)
        
        print(f"\n✓ Saved: {output_csv}")
        print(f"  Total reaches: {len(master_df)}")
        
        return master_df
    else:
        print("\n✗ No metrics found")
        return None


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Example 1: Compile all rivers from Greenberg dataset
    input_csv = r"E:\Dissertation\Data\GreenbergZhaoRees_river_datasheet.csv"
    output_csv = "E:/Dissertation/Data/RiverMapping/Mobility/master_mobility_metrics.csv"
    
    master_df = compile_mobility_metrics(input_csv, output_csv)
    
    # Example 2: Compile only specific rivers
    # problem_rivers = ['Ural_Kushum', 'Helmand_Kajaki', 'Chari_Ndjamena']
    # output_csv = "E:/Dissertation/Data/RiverMapping/Mobility/problem_rivers_mobility.csv"
    # compile_subset(input_csv, output_csv, river_subset=problem_rivers)
    
    # Example 3: Display first few rows
    if master_df is not None:
        print("\n" + "=" * 80)
        print("PREVIEW OF MASTER CSV (first 10 rows):")
        print(master_df.head(10).to_string(index=False))

Compiling mobility metrics from individual river CSVs...
✓ Aladan_VerkhoyanskiyPerevoz: 1 reaches
✓ Amazonas_Jatuarana: 1 reaches
✓ Amazonas_Tamshiyacu: 1 reaches
✓ AmuDarya_Kerki: 1 reaches
✓ Amur_Khabarovsk: 1 reaches
✓ Amur_Komsomolsk: 1 reaches
✓ Amyl_Kachulka: 1 reaches
✓ Apalachicola_NearBlountstown: 1 reaches
✓ Araguaia_Aruana: 1 reaches
✓ Araguaia_LuizAlves: 1 reaches
✓ Araguaia_SaoFelixDoAraguaia: 1 reaches
✓ Beas_MandiPlain: 1 reaches
✓ Benue_Ibi: 1 reaches
✓ Benue_Umaisha: 1 reaches
✓ Bhareli_NTRoadCrossing: 1 reaches
✓ BolshayaKet_Rodyonovka: 1 reaches
✓ Brahmaputra_Bahadurabad: 1 reaches
✓ Brahmaputra_Pandu: 1 reaches
✓ Brahmaputra_Pasighat: 1 reaches
✓ Brahmaputra_Yangcun: 1 reaches
✓ Chari_Bousso: 1 reaches
✓ Chari_Guelengdeng: 1 reaches
✓ Chari_Sahr: 1 reaches
✓ Chenab_Akhnoor: 1 reaches
✓ Cuiaba_PortoDoAlegre: 1 reaches
✓ Demini_PostoAjuricaba: 1 reaches
✓ Fraser_Hope: 1 reaches
✓ Gandak_Devghat: 1 reaches
✓ Gandak_Dumariaghat: 1 reaches
✓ Gandak_Triveni: 1 reaches
✓ G